# Full XGBoost Train And Evaluate

This notebook trains a full XGBoost model on the `v3` training split and evaluates it end-to-end on the full validation and test splits.

It combines:
- full-model training,
- full validation/test evaluation,
- naive baseline comparison,
- positive-track-only analysis,
- feature importance,
- optional SHAP on a sampled subset,
- and artifact export under a dedicated `xgboost_full` namespace.


In [ ]:
from pathlib import Path
import json
import os
import tempfile
import warnings

os.environ.setdefault("MPLCONFIGDIR", tempfile.mkdtemp(prefix="matplotlib-"))

import duckdb
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from IPython.display import display

try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError("Install xgboost first, e.g. `pip install -r requirements.txt`.") from exc

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "datasets").exists() and (candidate / "requirements.txt").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. Expected a parent containing 'datasets/' and 'requirements.txt'."
    )

ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / "datasets" / "processed" / "v3_features"
MODEL_DIR = ROOT / "artifacts" / "models" / "xgboost_full"
EVAL_DIR = ROOT / "artifacts" / "evaluations" / "xgboost_full"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.parquet"
VAL_PATH = DATA_DIR / "val.parquet"
TEST_PATH = DATA_DIR / "test.parquet"

assert TRAIN_PATH.exists(), f"Missing training split: {TRAIN_PATH}"
assert VAL_PATH.exists(), f"Missing validation split: {VAL_PATH}"
assert TEST_PATH.exists(), f"Missing test split: {TEST_PATH}"

RANDOM_STATE = 42
TOP_K = 5
RUN_FULL_SPLITS = True
DEBUG_MAX_TRAIN_TRACKS = None
DEBUG_MAX_VAL_TRACKS = None
DEBUG_MAX_TEST_TRACKS = None
RUN_SHAP = True
SHAP_SAMPLE_TRACKS = 1000

FEATURE_EXCLUDE = [
    "track_id",
    "observation_time",
    "target_country",
    "did_enter_within_60d",
    "days_to_entry",
]

TRAIN_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_TRAIN_TRACKS
VAL_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_VAL_TRACKS
TEST_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_TEST_TRACKS
RUN_MODE = "full" if RUN_FULL_SPLITS else "debug_sampled"

print({
    "run_mode": RUN_MODE,
    "top_k": TOP_K,
    "train_max_tracks": TRAIN_MAX_TRACKS,
    "val_max_tracks": VAL_MAX_TRACKS,
    "test_max_tracks": TEST_MAX_TRACKS,
    "run_shap": RUN_SHAP,
    "shap_sample_tracks": SHAP_SAMPLE_TRACKS,
})


In [ ]:
con = duckdb.connect()


def load_split(path: Path, max_tracks: int | None = None) -> pd.DataFrame:
    parquet_path = path.as_posix()
    if max_tracks is None:
        query = f"SELECT * FROM read_parquet('{parquet_path}')"
    else:
        query = f"""
            WITH sampled_tracks AS (
                SELECT track_id
                FROM read_parquet('{parquet_path}')
                GROUP BY track_id
                ORDER BY hash(track_id)
                LIMIT {int(max_tracks)}
            )
            SELECT d.*
            FROM read_parquet('{parquet_path}') d
            JOIN sampled_tracks st USING (track_id)
        """
    return con.execute(query).fetchdf()


def make_feature_matrix(df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series) -> pd.DataFrame:
    return df[feature_cols].copy().fillna(fill_values)


def binary_metrics(y_true: pd.Series, y_score: np.ndarray) -> dict:
    return {
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "average_precision": float(average_precision_score(y_true, y_score)),
        "log_loss": float(log_loss(y_true, y_score, labels=[0, 1])),
    }


def compute_candidate_stats(scored_df: pd.DataFrame) -> dict:
    per_track = scored_df.groupby("track_id").agg(
        candidates=("target_country", "size"),
        positives=("did_enter_within_60d", "sum"),
    )
    positive_mask = per_track["positives"] > 0
    return {
        "tracks": int(per_track.shape[0]),
        "positive_tracks": int(positive_mask.sum()),
        "avg_candidates_per_track": float(per_track["candidates"].mean()),
        "avg_future_countries_per_track": float(per_track["positives"].mean()),
        "avg_future_countries_per_positive_track": float(per_track.loc[positive_mask, "positives"].mean()) if positive_mask.any() else None,
    }


def ranking_metrics(scored_df: pd.DataFrame, k: int = 5) -> tuple[dict, pd.DataFrame]:
    rows = []
    for track_id, group in scored_df.groupby("track_id", sort=False):
        group = group.sort_values(["score", "tie_break"], ascending=[False, False]).reset_index(drop=True)
        labels = group["did_enter_within_60d"].to_numpy(dtype=int)
        positives = int(labels.sum())
        top = group.head(k)
        top_labels = top["did_enter_within_60d"].to_numpy(dtype=int)
        hits = int(top_labels.sum())

        precision = hits / k
        recall = hits / positives if positives else np.nan
        hit_rate = float(hits > 0) if positives else np.nan

        discounts = np.log2(np.arange(2, len(top_labels) + 2))
        dcg = float(((2 ** top_labels - 1) / discounts).sum())
        ideal = np.sort(labels)[::-1][: len(top_labels)]
        idcg = float(((2 ** ideal - 1) / np.log2(np.arange(2, len(ideal) + 2))).sum())
        ndcg = dcg / idcg if idcg > 0 else np.nan

        ap_accum = 0.0
        running_hits = 0
        for rank, rel in enumerate(top_labels, start=1):
            if rel:
                running_hits += 1
                ap_accum += running_hits / rank
        map_k = ap_accum / min(positives, k) if positives else np.nan

        rows.append(
            {
                "track_id": track_id,
                "positives": positives,
                "top_k_hits": hits,
                f"precision@{k}": precision,
                f"recall@{k}": recall,
                f"hit_rate@{k}": hit_rate,
                f"ndcg@{k}": ndcg,
                f"map@{k}": map_k,
            }
        )

    metric_df = pd.DataFrame(rows)
    positive_mask = metric_df["positives"] > 0
    summary = {
        "tracks": int(metric_df.shape[0]),
        "positive_tracks": int(positive_mask.sum()),
        f"precision@{k}": float(metric_df[f"precision@{k}"].mean()),
        f"recall@{k}": float(metric_df.loc[positive_mask, f"recall@{k}"].mean()) if positive_mask.any() else None,
        f"hit_rate@{k}": float(metric_df.loc[positive_mask, f"hit_rate@{k}"].mean()) if positive_mask.any() else None,
        f"ndcg@{k}": float(metric_df.loc[positive_mask, f"ndcg@{k}"].mean()) if positive_mask.any() else None,
        f"map@{k}": float(metric_df.loc[positive_mask, f"map@{k}"].mean()) if positive_mask.any() else None,
        "mean_future_countries_per_track": float(metric_df["positives"].mean()),
    }
    return summary, metric_df


def evaluate_predictions(scored_df: pd.DataFrame, k: int = 5) -> tuple[dict, pd.DataFrame]:
    candidate_stats = compute_candidate_stats(scored_df)
    binary = binary_metrics(scored_df["did_enter_within_60d"], scored_df["score"].to_numpy())
    ranking_all, track_metrics = ranking_metrics(scored_df, k=k)
    positive_track_metrics = track_metrics[track_metrics["positives"] > 0].copy()

    positive_summary = {
        "tracks": int(positive_track_metrics.shape[0]),
        "avg_future_countries_per_positive_track": float(positive_track_metrics["positives"].mean()) if not positive_track_metrics.empty else None,
        f"avg_top_{k}_hits_per_positive_track": float(positive_track_metrics["top_k_hits"].mean()) if not positive_track_metrics.empty else None,
        f"recall@{k}": float(positive_track_metrics[f"recall@{k}"].mean()) if not positive_track_metrics.empty else None,
        f"hit_rate@{k}": float(positive_track_metrics[f"hit_rate@{k}"].mean()) if not positive_track_metrics.empty else None,
        f"ndcg@{k}": float(positive_track_metrics[f"ndcg@{k}"].mean()) if not positive_track_metrics.empty else None,
        f"map@{k}": float(positive_track_metrics[f"map@{k}"].mean()) if not positive_track_metrics.empty else None,
    }

    return {
        "binary": binary,
        "candidate_stats": candidate_stats,
        "ranking_all_tracks": ranking_all,
        "ranking_positive_tracks": positive_summary,
    }, track_metrics


def score_xgboost(model: xgb.XGBClassifier, df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series) -> pd.DataFrame:
    X = make_feature_matrix(df, feature_cols, fill_values)
    scores = model.predict_proba(X)[:, 1]
    scored = df[["track_id", "observation_time", "target_country", "did_enter_within_60d", "days_to_entry", "target_avg_daily_streams", "target_new_entry_rate_30d"]].copy()
    scored["score"] = scores
    scored["tie_break"] = scored["target_new_entry_rate_30d"].fillna(0.0)
    scored["model_name"] = "xgboost"
    return scored


def score_naive_popularity(df: pd.DataFrame) -> pd.DataFrame:
    scored = df[["track_id", "observation_time", "target_country", "did_enter_within_60d", "days_to_entry", "target_avg_daily_streams", "target_new_entry_rate_30d"]].copy()
    primary = scored["target_avg_daily_streams"].fillna(0.0)
    tie_break = scored["target_new_entry_rate_30d"].fillna(0.0)
    raw_score = primary + tie_break * 1e-6
    score_min = float(raw_score.min())
    score_max = float(raw_score.max())
    if score_max > score_min:
        scored["score"] = (raw_score - score_min) / (score_max - score_min)
    else:
        scored["score"] = 0.5
    scored["tie_break"] = tie_break
    scored["model_name"] = "naive_popularity_baseline"
    return scored


def feature_category(name: str) -> str:
    if name.startswith("rank_") or name == "track_in_viral50_at_obs":
        return "current_footprint"
    if name.startswith("artist_") or name == "multi_artist_flag":
        return "artist_history"
    if name.startswith("target_"):
        return "target_country_priors"
    if name in {"cultural_dist_min", "cultural_dist_missing", "same_language_flag", "same_continent_flag", "neighbor_entered_count"}:
        return "origin_target_relationship"
    if name.startswith("af_") or name in {"duration_ms", "explicit", "days_since_release", "is_friday_release"}:
        return "audio_track_metadata"
    if name.startswith("observation_"):
        return "temporal"
    return "other"


In [ ]:
train_df = load_split(TRAIN_PATH, TRAIN_MAX_TRACKS)
val_df = load_split(VAL_PATH, VAL_MAX_TRACKS)
test_df = load_split(TEST_PATH, TEST_MAX_TRACKS)

feature_cols = [col for col in train_df.columns if col not in FEATURE_EXCLUDE]
fill_values = train_df[feature_cols].median(numeric_only=True)

X_train = make_feature_matrix(train_df, feature_cols, fill_values)
X_val = make_feature_matrix(val_df, feature_cols, fill_values)
X_test = make_feature_matrix(test_df, feature_cols, fill_values)
y_train = train_df["did_enter_within_60d"].astype(int)
y_val = val_df["did_enter_within_60d"].astype(int)
y_test = test_df["did_enter_within_60d"].astype(int)

negatives = int((y_train == 0).sum())
positives = int((y_train == 1).sum())
scale_pos_weight = negatives / max(positives, 1)

print(f"Train rows: {len(train_df):,} | tracks: {train_df['track_id'].nunique():,}")
print(f"Val rows:   {len(val_df):,} | tracks: {val_df['track_id'].nunique():,}")
print(f"Test rows:  {len(test_df):,} | tracks: {test_df['track_id'].nunique():,}")
print(f"Feature count: {len(feature_cols)}")
print(f"Training positives: {positives:,} ({positives / len(train_df) * 100:.2f}%)")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")


In [ ]:
model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric=["logloss", "aucpr"],
    tree_method="hist",
    n_estimators=800,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=50,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

best_iteration = getattr(model, "best_iteration", None)
best_score = getattr(model, "best_score", None)
print(f"Best iteration: {best_iteration}")
print(f"Best validation score: {best_score}")


In [ ]:
val_predictions = {
    "xgboost": score_xgboost(model, val_df, feature_cols, fill_values),
    "naive_popularity_baseline": score_naive_popularity(val_df),
}
test_predictions = {
    "xgboost": score_xgboost(model, test_df, feature_cols, fill_values),
    "naive_popularity_baseline": score_naive_popularity(test_df),
}

results = {"validation": {}, "test": {}}
track_metric_tables = {"validation": {}, "test": {}}
for split_name, pred_map in [("validation", val_predictions), ("test", test_predictions)]:
    for model_name, scored_df in pred_map.items():
        summary, track_metrics = evaluate_predictions(scored_df, k=TOP_K)
        results[split_name][model_name] = summary
        track_metric_tables[split_name][model_name] = track_metrics

comparison_rows = []
for split_name, model_map in results.items():
    for model_name, summary in model_map.items():
        comparison_rows.append({
            "split": split_name,
            "model": model_name,
            "roc_auc": summary["binary"]["roc_auc"],
            "average_precision": summary["binary"]["average_precision"],
            "log_loss": summary["binary"]["log_loss"],
            f"precision@{TOP_K}": summary["ranking_all_tracks"][f"precision@{TOP_K}"],
            f"recall@{TOP_K}": summary["ranking_all_tracks"][f"recall@{TOP_K}"],
            f"hit_rate@{TOP_K}": summary["ranking_all_tracks"][f"hit_rate@{TOP_K}"],
            f"ndcg@{TOP_K}": summary["ranking_all_tracks"][f"ndcg@{TOP_K}"],
            f"map@{TOP_K}": summary["ranking_all_tracks"][f"map@{TOP_K}"],
            f"positive_recall@{TOP_K}": summary["ranking_positive_tracks"][f"recall@{TOP_K}"],
            f"positive_hit_rate@{TOP_K}": summary["ranking_positive_tracks"][f"hit_rate@{TOP_K}"],
            f"positive_ndcg@{TOP_K}": summary["ranking_positive_tracks"][f"ndcg@{TOP_K}"],
            f"positive_map@{TOP_K}": summary["ranking_positive_tracks"][f"map@{TOP_K}"],
            "tracks": summary["candidate_stats"]["tracks"],
            "positive_tracks": summary["candidate_stats"]["positive_tracks"],
            "avg_candidates_per_track": summary["candidate_stats"]["avg_candidates_per_track"],
        })
comparison_df = pd.DataFrame(comparison_rows).sort_values(["split", "model"]).reset_index(drop=True)
print("Model vs baseline comparison:")
display(comparison_df)

positive_analysis_rows = []
for split_name, model_map in results.items():
    for model_name, summary in model_map.items():
        positive_analysis_rows.append({
            "split": split_name,
            "model": model_name,
            "positive_tracks": summary["ranking_positive_tracks"]["tracks"],
            "avg_future_countries_per_positive_track": summary["ranking_positive_tracks"]["avg_future_countries_per_positive_track"],
            f"avg_top_{TOP_K}_hits_per_positive_track": summary["ranking_positive_tracks"][f"avg_top_{TOP_K}_hits_per_positive_track"],
            f"recall@{TOP_K}": summary["ranking_positive_tracks"][f"recall@{TOP_K}"],
            f"hit_rate@{TOP_K}": summary["ranking_positive_tracks"][f"hit_rate@{TOP_K}"],
            f"ndcg@{TOP_K}": summary["ranking_positive_tracks"][f"ndcg@{TOP_K}"],
            f"map@{TOP_K}": summary["ranking_positive_tracks"][f"map@{TOP_K}"],
        })
positive_analysis_df = pd.DataFrame(positive_analysis_rows).sort_values(["split", "model"]).reset_index(drop=True)
print()
print("Positive-track-only view:")
display(positive_analysis_df)

interpretation_df = positive_analysis_df[["split", "model", f"avg_top_{TOP_K}_hits_per_positive_track", f"hit_rate@{TOP_K}", f"recall@{TOP_K}"]].copy()
interpretation_df["interpretation"] = interpretation_df.apply(
    lambda row: f"On positive tracks, top-{TOP_K} returns about {row[f'avg_top_{TOP_K}_hits_per_positive_track']:.2f} correct countries on average.",
    axis=1,
)
print()
print("Interpretation table:")
display(interpretation_df)

evaluation_summary = {
    "config": {
        "top_k": TOP_K,
        "run_mode": RUN_MODE,
        "run_full_splits": RUN_FULL_SPLITS,
        "train_max_tracks": TRAIN_MAX_TRACKS,
        "val_max_tracks": VAL_MAX_TRACKS,
        "test_max_tracks": TEST_MAX_TRACKS,
        "run_shap": RUN_SHAP,
        "shap_sample_tracks": SHAP_SAMPLE_TRACKS,
    },
    "validation": results["validation"],
    "test": results["test"],
}


In [ ]:
booster = model.get_booster()
importance_gain = booster.get_score(importance_type="gain")
importance_weight = booster.get_score(importance_type="weight")

importance_df = pd.DataFrame({"feature": feature_cols})
importance_df["gain"] = importance_df["feature"].map(importance_gain).fillna(0.0)
importance_df["weight"] = importance_df["feature"].map(importance_weight).fillna(0.0)
importance_df["category"] = importance_df["feature"].map(feature_category)
importance_df = importance_df.sort_values(["gain", "weight"], ascending=[False, False]).reset_index(drop=True)

print("Top 20 features by gain:")
display(importance_df.head(20))

try:
    fig, axes = plt.subplots(1, 2, figsize=(16, 10))
    plot_gain = importance_df.head(20).sort_values("gain")
    plot_weight = importance_df.sort_values("weight", ascending=False).head(20).sort_values("weight")
    axes[0].barh(plot_gain["feature"], plot_gain["gain"])
    axes[0].set_title("Top 20 Feature Importance (gain)")
    axes[0].set_xlabel("gain")
    axes[1].barh(plot_weight["feature"], plot_weight["weight"])
    axes[1].set_title("Top 20 Feature Importance (weight)")
    axes[1].set_xlabel("weight")
    plt.tight_layout()
    plt.show()
    plt.close(fig)
except Exception as exc:
    print(f"Skipping feature-importance plots after runtime error: {exc}")

category_summary = (
    importance_df.groupby("category")[["gain", "weight"]]
    .sum()
    .sort_values("gain", ascending=False)
)
print("Feature category summary:")
display(category_summary)


In [ ]:
shap_available = False
shap_summary_df = None
shap_message = None

if RUN_SHAP:
    try:
        import shap
        shap_available = True
    except ImportError:
        shap_message = "SHAP is not installed. Run `pip install -r requirements.txt` and re-run this section."
        print(shap_message)

if shap_available:
    try:
        shap_source_df = test_df if len(test_df) <= len(val_df) else val_df
        shap_track_limit = min(SHAP_SAMPLE_TRACKS, shap_source_df["track_id"].nunique())
        shap_track_ids = (
            shap_source_df[["track_id"]]
            .drop_duplicates()
            .sort_values("track_id")
            .head(shap_track_limit)["track_id"]
            .tolist()
        )
        shap_df = shap_source_df[shap_source_df["track_id"].isin(shap_track_ids)].copy()
        X_shap = make_feature_matrix(shap_df, feature_cols, fill_values)

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_shap)
        if isinstance(shap_values, list):
            shap_values = shap_values[-1]

        shap_summary_df = pd.DataFrame({
            "feature": feature_cols,
            "mean_abs_shap": np.abs(shap_values).mean(axis=0),
        }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
        shap_summary_df["category"] = shap_summary_df["feature"].map(feature_category)

        print(f"Computed SHAP on {len(shap_df):,} rows from {len(shap_track_ids):,} sampled tracks.")
        display(shap_summary_df.head(20))

        shap.summary_plot(shap_values, X_shap, show=False, max_display=20)
        plt.title("SHAP Summary Plot")
        plt.tight_layout()
        plt.show()

        shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False, max_display=20)
        plt.title("Mean Absolute SHAP Values")
        plt.tight_layout()
        plt.show()

        positive_candidates = shap_df[shap_df["did_enter_within_60d"] == 1]
        local_idx = positive_candidates.index[0] if not positive_candidates.empty else shap_df.index[0]
        local_pos = shap_df.index.get_loc(local_idx)
        expected_value = explainer.expected_value
        if isinstance(expected_value, (list, np.ndarray)):
            expected_value = np.asarray(expected_value).reshape(-1)[-1]
        print("Local SHAP explanation row:")
        display(shap_df.loc[[local_idx], ["track_id", "target_country", "did_enter_within_60d", "days_to_entry"]])
        shap.plots._waterfall.waterfall_legacy(
            expected_value,
            shap_values[local_pos],
            feature_names=feature_cols,
            max_display=15,
        )
        plt.show()
    except Exception as exc:
        shap_summary_df = None
        shap_message = f"Skipping SHAP section after runtime error: {exc}"
        print(shap_message)
else:
    print("Skipping SHAP section.")


In [ ]:
training_summary = {
    "data": {
        "train_path": str(TRAIN_PATH),
        "val_path": str(VAL_PATH),
        "test_path": str(TEST_PATH),
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "train_tracks": int(train_df["track_id"].nunique()),
        "val_tracks": int(val_df["track_id"].nunique()),
        "test_tracks": int(test_df["track_id"].nunique()),
    },
    "model": {
        "type": "xgboost.XGBClassifier",
        "top_k": TOP_K,
        "best_iteration": None if best_iteration is None else int(best_iteration),
        "best_score": None if best_score is None else float(best_score),
        "scale_pos_weight": float(scale_pos_weight),
        "feature_count": len(feature_cols),
        "random_state": RANDOM_STATE,
    },
    "config": {
        "run_mode": RUN_MODE,
        "run_full_splits": RUN_FULL_SPLITS,
        "train_max_tracks": TRAIN_MAX_TRACKS,
        "val_max_tracks": VAL_MAX_TRACKS,
        "test_max_tracks": TEST_MAX_TRACKS,
        "run_shap": RUN_SHAP,
        "shap_sample_tracks": SHAP_SAMPLE_TRACKS,
    },
    "feature_cols": feature_cols,
    "train_fill_values": {col: float(fill_values[col]) for col in feature_cols},
}

model_path = MODEL_DIR / "xgb_classifier.json"
training_summary_path = MODEL_DIR / "training_summary.json"
evaluation_summary_path = EVAL_DIR / "evaluation_summary.json"
comparison_path = EVAL_DIR / "model_vs_baseline_comparison.parquet"
positive_analysis_path = EVAL_DIR / "positive_track_analysis.parquet"
importance_path = EVAL_DIR / "feature_importance.parquet"
val_preds_path = EVAL_DIR / "val_predictions.parquet"
test_preds_path = EVAL_DIR / "test_predictions.parquet"
val_track_metrics_path = EVAL_DIR / "val_track_metrics.parquet"
test_track_metrics_path = EVAL_DIR / "test_track_metrics.parquet"
val_positive_track_metrics_path = EVAL_DIR / "val_track_metrics_positive_only.parquet"
test_positive_track_metrics_path = EVAL_DIR / "test_track_metrics_positive_only.parquet"
shap_summary_path = EVAL_DIR / "shap_summary.parquet"

model.save_model(model_path)
with open(training_summary_path, "w", encoding="utf-8") as f:
    json.dump(training_summary, f, indent=2)
with open(evaluation_summary_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, indent=2)

con.register("comparison_df", comparison_df)
con.register("positive_analysis_df", positive_analysis_df)
con.register("importance_df", importance_df)
con.register("val_scored_df", val_predictions["xgboost"])
con.register("test_scored_df", test_predictions["xgboost"])
con.register("val_track_metrics_df", track_metric_tables["validation"]["xgboost"])
con.register("test_track_metrics_df", track_metric_tables["test"]["xgboost"])
con.register("val_positive_track_metrics_df", track_metric_tables["validation"]["xgboost"][track_metric_tables["validation"]["xgboost"]["positives"] > 0])
con.register("test_positive_track_metrics_df", track_metric_tables["test"]["xgboost"][track_metric_tables["test"]["xgboost"]["positives"] > 0])

con.execute(f"COPY comparison_df TO '{comparison_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY positive_analysis_df TO '{positive_analysis_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY importance_df TO '{importance_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY val_scored_df TO '{val_preds_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY test_scored_df TO '{test_preds_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY val_track_metrics_df TO '{val_track_metrics_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY test_track_metrics_df TO '{test_track_metrics_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY val_positive_track_metrics_df TO '{val_positive_track_metrics_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY test_positive_track_metrics_df TO '{test_positive_track_metrics_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")

if shap_summary_df is not None:
    con.register("shap_summary_df", shap_summary_df)
    con.execute(f"COPY shap_summary_df TO '{shap_summary_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
    con.unregister("shap_summary_df")

for name in [
    "comparison_df",
    "positive_analysis_df",
    "importance_df",
    "val_scored_df",
    "test_scored_df",
    "val_track_metrics_df",
    "test_track_metrics_df",
    "val_positive_track_metrics_df",
    "test_positive_track_metrics_df",
]:
    con.unregister(name)

print(f"Saved model to: {model_path}")
print(f"Saved training summary to: {training_summary_path}")
print(f"Saved evaluation summary to: {evaluation_summary_path}")
print(f"Saved comparison table to: {comparison_path}")
print(f"Saved positive-track analysis to: {positive_analysis_path}")
print(f"Saved feature importance to: {importance_path}")
if shap_summary_df is not None:
    print(f"Saved SHAP summary to: {shap_summary_path}")
